# Validate VIIRS pixel-code legends from the NOAA TIFF metadata

**Goal.** Treat the embedded NOAA GeoTIFF metadata as the **source of truth** for VIIRS pixel
codes, and cross-check it against:

1. The **Atlantis processor** (`src/atlantis/fetchers/viirs/processor.py`) — what the codebase
   currently *assumes*.
2. The **VIIRS VFM ATBD v1.0** (June 2021) Table 2-4 / 2-5 — bundled at
   `docs/viirs_atbd_pdftotext.txt`.
3. Our previously-scraped notes in `tmp/source_layers_comparison.md` and
   `/memories/repo/viirs-public-codes.md`.

Atlantis declares two VIIRS backends in `src/atlantis/fetchers/viirs/backend.py`:

| Backend id     | Class             | Source URL pattern                                                                    | Filename pattern              |
| -------------- | ----------------- | ------------------------------------------------------------------------------------- | ----------------------------- |
| `noaa_s3`      | `NoaaS3Backend`   | `s3://noaa-jpss/JPSS_Blended_Products/VFM_1day_GLB/TIF/YYYY/MM/DD/`                   | `VIIRS-Flood-1day-GLB{aoi}_*` |
| `gmu_legacy`   | `GmuLegacyBackend`| `https://jpssflood.gmu.edu/downloads/pub/YYYYMMDD/tif/`                               | `*_005day_{aoi}.tif`          |

**This notebook focuses on `noaa_s3` only.** The GMU legacy server has been unavailable for an
extended period (its `/downloads/pub` subtree returns 503; we tracked that on GitHub) and is out of
scope here. Whenever the GMU server returns we can re-enable that branch by un-skipping the
backend loop.

We use Typhoon Vamco (Luzon, Philippines, **2020-11-13**) because:

- It is a documented Atlantis test event with known harmonised outputs in `data/Vamco_2020/`.
- The bbox covers a single VIIRS AOI tile, keeping the network footprint small.

> Internet access is required — every raw read in this notebook is via `/vsicurl/`. No bytes are
> written to disk; we stream small windows.

## 1. Setup

Resolve the workspace root from this notebook's location, then ensure the editable `atlantis`
package is on `sys.path`.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
# Walk up until we find the repo marker (pyproject.toml).
candidate = NOTEBOOK_DIR
while candidate != candidate.parent and not (candidate / 'pyproject.toml').exists():
    candidate = candidate.parent
REPO_ROOT = candidate
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'notebook  = {NOTEBOOK_DIR.relative_to(REPO_ROOT)}')

# Make GDAL behave well with /vsicurl/ over slow links.
os.environ.setdefault('GDAL_HTTP_TIMEOUT', '30')
os.environ.setdefault('GDAL_HTTP_MAX_RETRY', '3')
os.environ.setdefault('CPL_VSIL_CURL_CHUNK_SIZE', '524288')  # 512 KiB

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

from atlantis.fetchers.viirs import VIIRSFetcher
from atlantis.fetchers.viirs.backend import (
    ListingLocation,
    NoaaS3Backend,
    get_backend,
    list_backends,
)
from atlantis.fetchers.viirs.processor import (
    CLOUD_CODES,
    FILL_CODES,
    FLOOD_MIN_CODE,
    OPEN_WATER_CODES,
    PERMANENT_WATER_CODES,
    SEASONAL_WATER_CODES,
)

ACTIVE_BACKENDS = ['noaa_s3']  # gmu_legacy is offline; tracked on GitHub.
print('Atlantis VIIRS backends declared :', list_backends())
print('Backends used in this notebook   :', ACTIVE_BACKENDS)

## 2. Three reference legends, side-by-side

Before we touch the network we capture the three reference legends as Python tables so we can
diff them against the **authoritative legend** that the live NOAA TIFF tags will give us further
down. None of these three are the source of truth — they're what the codebase / docs / notes
currently *claim*.

- **ATBD legend** — VIIRS VFM ATBD v1.0 Table 2-4 / 2-5 (`docs/viirs_atbd_pdftotext.txt`).
- **Pre-notebook notes legend** — a snapshot of what `tmp/source_layers_comparison.md` and
  `/memories/repo/viirs-public-codes.md` claim about the public NOAA TIFF (used for drift
  detection only).
- **Atlantis decoder** — pulled directly from the live constants in
  `atlantis.fetchers.viirs.processor`.

In [ ]:
# --- Reference 1: ATBD legend (Tables 2-4 / 2-5 in docs/viirs_atbd_pdftotext.txt) ---
ATBD_LEGEND: dict[int, str] = {
    0:   'Insufficient data',
    1:   'No data',
    3:   'Clear-sky land (no water)',
    7:   'Inland water (lake/river ref mask)',
    17:  'Vegetation',
    20:  'Snow / ice',
    27:  'Bad input',
    30:  'Cloud',
    40:  'Cloud shadow',
    50:  'Terrain shadow',
    100: 'Open normal water',
}
# Per ATBD: codes 101..200 encode flood-water fraction as (code-100)%.
ATBD_FRAC_RANGE = (101, 200)

# --- Reference 2: pre-notebook NOTES snapshot of what the public TIFF says ---
# Snapshot of what tmp/source_layers_comparison.md and /memories/repo/viirs-public-codes.md
# claimed BEFORE this notebook ran. Kept static so the cross-check can flag drift between
# our notes and what the live TIFF tag actually reports — the live tag wins.
PUBLIC_TIFF_LEGEND: dict[int, str] = {
    0:   'Insufficient data',
    1:   'No data',
    3:   'Clear-sky land',
    15:  'Floodwater without fraction retrieval',
    17:  'Vegetation',
    20:  'Snow / ice',
    27:  'Bad input',
    30:  'Cloud',
    40:  'Cloud shadow',
    50:  'Terrain shadow',
    99:  'NormalWater (open reference water)',
}
PUBLIC_TIFF_FRAC_RANGE = (100, 200)

# --- Reference 3: live Atlantis decoder constants ---
ATLANTIS_DECODER = {
    'FILL_CODES':            sorted(FILL_CODES),
    'CLOUD_CODES':           sorted(CLOUD_CODES),
    'PERMANENT_WATER_CODES': sorted(PERMANENT_WATER_CODES),
    'SEASONAL_WATER_CODES':  sorted(SEASONAL_WATER_CODES),
    'OPEN_WATER_CODES':      sorted(OPEN_WATER_CODES),
    'FLOOD_FRACTION_RANGE':  (101, 200),  # hardcoded in processor.py
    'FLOOD_MIN_CODE_default': FLOOD_MIN_CODE,
}
for k, v in ATLANTIS_DECODER.items():
    print(f'{k:>26}  {v}')

In [ ]:
# Build a single comparison DataFrame across all integer codes seen in any legend.
all_codes = sorted(set(ATBD_LEGEND) | set(PUBLIC_TIFF_LEGEND))

def atlantis_meaning(code: int) -> str:
    if code in FILL_CODES:
        return 'fill -> quality_mask=0'
    if code in CLOUD_CODES:
        return 'cloud -> quality_mask=0'
    if code in PERMANENT_WATER_CODES:
        return 'permanent_water=1'
    if code in SEASONAL_WATER_CODES:
        return '(seasonal — declared but not written)'
    if code in OPEN_WATER_CODES:
        return '(open water — declared but not written)'
    return '— (passes through as 0.0 flood_fraction)'

legend_df = pd.DataFrame(
    [
        {
            'code': c,
            'ATBD (bundled)':       ATBD_LEGEND.get(c, ''),
            'Public TIFF (NOAA)':   PUBLIC_TIFF_LEGEND.get(c, ''),
            'Atlantis processor':   atlantis_meaning(c),
        }
        for c in all_codes
    ]
).set_index('code')
legend_df

Notice already, *before any network call*, that:

- Code **17** is `Vegetation` in both references but Atlantis treats it as `permanent_water`.
- Code **99** is meaningful (`NormalWater`) only in the public TIFF table; Atlantis declares
  `OPEN_WATER_CODES = {99}` but never writes it to any output layer.
- Code **100** is real flood data in the public TIFF range but is **excluded** from Atlantis'
  flood-fraction window `[101, 200]`.

## 3. Pick the event and discover the AOI tile to probe

We let the `VIIRSFetcher` itself tell us which packaged AOI(s) intersect the Vamco bbox — this is
the same code that decides what to download in production. We then keep the smallest AOI ID for
deterministic remote naming.

In [ ]:
from atlantis.models.event import FloodEvent
from datetime import date

event = FloodEvent(
    event_id='Vamco_2020_legend_probe',
    bbox=(121.14, 16.72, 122.25, 18.45),
    start_date=date(2020, 11, 13),
    end_date=date(2020, 11, 13),
)
probe_date = datetime(2020, 11, 13, tzinfo=timezone.utc)

# Use NoaaS3Backend's fetcher to enumerate AOIs (the GeoJSON is identical regardless of backend).
probe_fetcher = VIIRSFetcher(backend='noaa_s3', data_format='tif')
aois = probe_fetcher._intersecting_aois(event)
print(f'Intersecting AOIs for Vamco bbox ({len(aois)} hit):')
print(aois[['AOI_ID', 'geometry']].head().to_string(index=False))

AOI_ID = int(aois.iloc[0]['AOI_ID'])
print(f'\nUsing AOI_ID = {AOI_ID:03d} for the rest of this notebook.')

## 4. Resolve one remote raw TIFF per backend

For each backend we call the same methods the production fetcher uses:

1. `get_listing_location(...)` — derive the listing prefix / URL for the date.
2. `get_directory_links(...)` — list candidate filenames.
3. `find_remote_filename(aoi_id, ...)` — pick the file for our AOI.
4. `build_result_url(...)` — assemble the GET-able URL.

We **do not download** anything; we keep the URL and read it via `/vsicurl/`.

In [ ]:
import time

BACKEND_BASE_URLS = {
    'noaa_s3': 'https://noaa-jpss.s3.amazonaws.com',
}
TIMEOUT = 30  # seconds per HTTP call

# Candidate dates to try in order. Useful for transient 5xx flutters.
PROBE_DATE_CANDIDATES = [
    datetime(2020, 11, 13, tzinfo=timezone.utc),
    datetime(2020, 11, 12, tzinfo=timezone.utc),
    datetime(2020, 11, 14, tzinfo=timezone.utc),
]

def _try_listing(backend, base_url: str, dt: datetime, retries: int = 2) -> tuple[ListingLocation, list[str], Exception | None]:
    loc = backend.get_listing_location(base_url, dt, 'tif')
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            entries = backend.get_directory_links(base_url, loc.locator, TIMEOUT)
            return loc, entries, None
        except Exception as exc:  # noqa: BLE001
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    return loc, [], last_exc

def resolve_remote_url(backend_id: str) -> tuple[str | None, ListingLocation | None, list[str], datetime | None]:
    """Return (vsicurl_url, listing_location, entries, date_used) for the AOI."""
    backend = get_backend(backend_id)
    base_url = BACKEND_BASE_URLS[backend_id]
    last_loc: ListingLocation | None = None
    last_entries: list[str] = []
    for dt in PROBE_DATE_CANDIDATES:
        loc, entries, exc = _try_listing(backend, base_url, dt)
        last_loc, last_entries = loc, entries
        if exc is not None:
            print(f'[{backend_id}] {dt.date()} listing failed: {exc}')
            continue
        fname = backend.find_remote_filename(AOI_ID, entries)
        if fname is None:
            print(f'[{backend_id}] {dt.date()} no AOI {AOI_ID:03d} match in {len(entries)} entries')
            continue
        url = backend.build_result_url(base_url, loc.locator, fname)
        return url, loc, entries, dt
    return None, last_loc, last_entries, None

backend_urls: dict[str, str | None] = {}
backend_dates: dict[str, datetime | None] = {}
for bid in ACTIVE_BACKENDS:
    url, loc, entries, dt = resolve_remote_url(bid)
    backend_urls[bid] = url
    backend_dates[bid] = dt
    print(f'\n--- {bid} ---')
    print(f'  listing locator : {loc.locator if loc else "n/a"}')
    print(f'  entries found   : {len(entries)}')
    print(f'  date used       : {dt.date() if dt else "—"}')
    print(f'  resolved URL    : {url}')

## 5. Read embedded metadata from each remote TIFF

Each NOAA VIIRS TIFF carries a structured metadata block (TIFF `IMAGEDESCRIPTION` /
`GDAL_METADATA` tags) that documents the pixel-code legend at the source. We pull this *without*
downloading the image bytes — `rasterio.open(/vsicurl/...)` only fetches the header range.

In [ ]:
def vsicurl(url: str) -> str:
    return url if url.startswith('/vsicurl/') else f'/vsicurl/{url}'

metadata_tags: dict[str, dict[str, object]] = {}
for bid, url in backend_urls.items():
    if not url:
        print(f'[{bid}] skipped — no URL')
        continue
    try:
        with rasterio.open(vsicurl(url)) as ds:
            metadata_tags[bid] = {
                'driver':       ds.driver,
                'dtype':        ds.dtypes[0],
                'count':        ds.count,
                'width':        ds.width,
                'height':       ds.height,
                'crs':          str(ds.crs),
                'transform':    tuple(ds.transform)[:6],
                'nodata':       ds.nodata,
                'tags':         ds.tags(),
                'band1_tags':   ds.tags(1),
                'descriptions': ds.descriptions,
            }
    except Exception as exc:  # noqa: BLE001
        print(f'[{bid}] rasterio open failed: {exc}')
        metadata_tags[bid] = {'error': str(exc)}

for bid, meta in metadata_tags.items():
    print(f'\n===== {bid} =====')
    for k, v in meta.items():
        if k == 'tags' and isinstance(v, dict):
            # Pretty-print tags so legend strings are readable.
            print(f'  {k}:')
            for tk, tv in v.items():
                preview = tv if len(str(tv)) < 200 else str(tv)[:200] + ' …'
                print(f'    - {tk}: {preview}')
        else:
            print(f'  {k}: {v}')

In [ ]:
import re

# Match either `100-200: Water Fractions` (range) or `17: Vegetation` (single).
_RANGE_PATTERN  = re.compile(r'(\d+)\s*-\s*(\d+)\s*[:=]\s*([^,;\n]+)')
_SINGLE_PATTERN = re.compile(r'(?<![\d-])(\d+)\s*[:=]\s*([^,;\n]+)')

# Tag names that, by NOAA convention, carry the pixel-code legend. Restricting
# parsing to these avoids picking up timestamp / coordinate strings.
LEGEND_TAG_KEYS = {
    'TypeDescription',
    'WaterDetection#TypeDescription',
    'long_name',
}

def extract_embedded_legend(meta):
    # singles -> {code: label}; ranges -> [(lo, hi, label), ...]
    singles: dict[int, str] = {}
    ranges: list[tuple[int, int, str]] = []
    blobs: list[str] = []
    for src_key in ('tags', 'band1_tags'):
        src = meta.get(src_key)
        if not isinstance(src, dict):
            continue
        for tk, tv in src.items():
            if isinstance(tv, str) and any(key in tk for key in LEGEND_TAG_KEYS):
                blobs.append(tv)
    seen_range_spans: set[tuple[int, int]] = set()
    for blob in blobs:
        consumed: list[tuple[int, int]] = []
        for m in _RANGE_PATTERN.finditer(blob):
            lo, hi = int(m.group(1)), int(m.group(2))
            label = m.group(3).strip().rstrip('.').strip()
            if (lo, hi) not in seen_range_spans:
                ranges.append((lo, hi, label))
                seen_range_spans.add((lo, hi))
            consumed.append(m.span())

        def _is_consumed(pos: int) -> bool:
            return any(start <= pos < end for start, end in consumed)

        for m in _SINGLE_PATTERN.finditer(blob):
            if _is_consumed(m.start()):
                continue
            try:
                code = int(m.group(1))
            except ValueError:
                continue
            if 0 <= code <= 255:
                label = m.group(2).strip().rstrip('.').strip()
                singles.setdefault(code, label)
    return dict(sorted(singles.items())), ranges

embedded_legends: dict[str, dict[int, str]] = {}
embedded_ranges:  dict[str, list[tuple[int, int, str]]] = {}
for bid, meta in metadata_tags.items():
    if meta.get('error'):
        embedded_legends[bid] = {}
        embedded_ranges[bid] = []
        continue
    singles, ranges = extract_embedded_legend(meta)
    embedded_legends[bid] = singles
    embedded_ranges[bid]  = ranges
    print(f'\n[{bid}] embedded singles ({len(singles)} codes):')
    for c, lbl in singles.items():
        print(f'  {c:>3}: {lbl}')
    if ranges:
        print(f'[{bid}] embedded ranges ({len(ranges)}):')
        for lo, hi, lbl in ranges:
            print(f'  {lo:>3}..{hi:<3}: {lbl}')

## 6. Sample actual pixel values

Header metadata tells us what codes *should* appear; we now read a windowed slab from each remote
TIFF to confirm what codes are *actually present* in a real granule.

We deliberately read a **center-clip** rather than the full granule — every backend's tile is
thousands of pixels wide and a 1024×1024 sample is enough to count classes.

In [ ]:
from rasterio.windows import from_bounds

EVENT_BBOX = event.bbox  # (minx, miny, maxx, maxy) in EPSG:4326

def sample_codes(url: str) -> pd.Series:
    """Read pixels inside the event bbox (land + flood, not open ocean centre)."""
    with rasterio.open(vsicurl(url)) as ds:
        win = from_bounds(*EVENT_BBOX, transform=ds.transform).round_offsets().round_lengths()
        # Clip to dataset extent.
        win = win.intersection(Window(0, 0, ds.width, ds.height))
        arr = ds.read(1, window=win)
    values, counts = np.unique(arr, return_counts=True)
    s = pd.Series(counts, index=values, name='count').sort_index()
    s.index.name = 'code'
    return s

code_samples: dict[str, pd.Series] = {}
for bid, url in backend_urls.items():
    if not url:
        continue
    try:
        code_samples[bid] = sample_codes(url)
        print(f'[{bid}] {len(code_samples[bid])} distinct codes in Vamco bbox window')
    except Exception as exc:  # noqa: BLE001
        print(f'[{bid}] sampling failed: {exc}')

if code_samples:
    samples_df = pd.concat(code_samples, axis=1).fillna(0).astype('int64')
    samples_df.columns = [f'{bid}__count' for bid in samples_df.columns]
    # Also show a percentage column to make the distribution legible.
    for bid in code_samples:
        col = f'{bid}__count'
        samples_df[f'{bid}__pct'] = (samples_df[col] / samples_df[col].sum() * 100).round(2)
    display(samples_df)

## 7. Master cross-check table

For every code that appears in **any** legend (ATBD, public TIFF, embedded TIFF tags) or shows
up in **any** of the sampled rasters, we assemble a single row. This is the artefact you can
lift verbatim into a doc or a fix-it PR.

In [ ]:
def _embedded_label(bid: str, code: int) -> str:
    singles = embedded_legends.get(bid, {})
    if code in singles:
        return singles[code]
    for lo, hi, lbl in embedded_ranges.get(bid, []):
        if lo <= code <= hi:
            return f'(range {lo}-{hi}) {lbl}'
    return ''

# Build the universe of *classification* codes only (codes 0..99 plus declared
# extras). Codes in the 100-200 fraction band are summarised in a single row.
classification_codes = set(c for c in (set(ATBD_LEGEND) | set(PUBLIC_TIFF_LEGEND)) if c < 100)
for legend in embedded_legends.values():
    classification_codes |= {c for c in legend if c < 100}
for s in code_samples.values():
    classification_codes |= {int(c) for c in s.index if int(c) < 100}

def in_flood_range(code: int) -> str:
    notes = []
    if ATBD_FRAC_RANGE[0] <= code <= ATBD_FRAC_RANGE[1]: notes.append('ATBD frac')
    if PUBLIC_TIFF_FRAC_RANGE[0] <= code <= PUBLIC_TIFF_FRAC_RANGE[1]: notes.append('public frac')
    if 101 <= code <= 200: notes.append('Atlantis frac')
    return ', '.join(notes)

rows = []
for code in sorted(classification_codes):
    row = {
        'code':                code,
        'ATBD':                ATBD_LEGEND.get(code, ''),
        'Notes (pre-NB)':      PUBLIC_TIFF_LEGEND.get(code, ''),
        'Atlantis decoder':    atlantis_meaning(code),
    }
    for bid in ACTIVE_BACKENDS:
        row[f'embedded [{bid}]'] = _embedded_label(bid, code)
        s = code_samples.get(bid)
        row[f'count [{bid}]']   = int(s.get(code, 0)) if s is not None else 0
    rows.append(row)

# Collapse the 100-200 band into one summary row per backend.
def _frac_summary(bid: str) -> int:
    s = code_samples.get(bid)
    if s is None:
        return 0
    in_band = s[(s.index >= 100) & (s.index <= 200)]
    return int(in_band.sum())

frac_row = {
    'code':                '100-200',
    'ATBD':                '101-200 = flood-water fraction (code-100)%',
    'Notes (pre-NB)':      '100-200 = Water Fractions',
    'Atlantis decoder':    'flood_fraction = (code-100)/100 if 101<=code<=200 else 0',
}
for bid in ACTIVE_BACKENDS:
    frac_row[f'embedded [{bid}]'] = (
        next((f'(range {lo}-{hi}) {lbl}' for lo, hi, lbl in embedded_ranges.get(bid, [])), '')
    )
    frac_row[f'count [{bid}]'] = _frac_summary(bid)
rows.append(frac_row)

crosscheck_df = pd.DataFrame(rows).set_index('code')

# Keep only rows that have at least one label or one nonzero count.
def _row_has_signal(r: pd.Series) -> bool:
    has_label = any(bool(r[col]) for col in r.index if isinstance(r[col], str))
    has_count = any(int(r[col]) > 0 for col in r.index if col.startswith('count ['))
    return has_label or has_count

with pd.option_context('display.max_rows', None, 'display.max_colwidth', 60):
    display(crosscheck_df[crosscheck_df.apply(_row_has_signal, axis=1)])

## 8. Programmatic findings

Rather than hand-write conclusions, we let the data flag the issues so this notebook re-runs
deterministically against new dates / backends.

In [ ]:
findings: list[str] = []

# The embedded NOAA TIFF metadata is treated as the source of truth from here on.
# Each finding below quotes the live tag and contrasts it with Atlantis' assumptions.

# F1 — Atlantis decodes code 17 as permanent_water; embedded metadata says 'Vegetation'.
for bid, leg in embedded_legends.items():
    if 17 in leg and 'veg' in leg[17].lower() and 17 in PERMANENT_WATER_CODES:
        findings.append(
            f'F1: Embedded NOAA TIFF metadata labels code 17 as {leg[17]!r}, but Atlantis maps '
            f'17 -> permanent_water. The current permanent_water output is not trustworthy.'
        )
        break

# F2 — Atlantis flood-fraction window misses code 100 declared by the embedded range.
for bid, ranges in embedded_ranges.items():
    for lo, hi, lbl in ranges:
        if lo == 100 and 'fraction' in lbl.lower():
            findings.append(
                f'F2: Embedded NOAA TIFF metadata declares the water-fraction range as '
                f'{lo}-{hi} ({lbl!r}), but Atlantis only decodes [101, 200], silently dropping '
                f'code 100 as 0.0.'
            )
            break

# F3 — Code 99 is meaningful in the embedded legend but Atlantis never propagates it.
any_99_seen = any(99 in s.index for s in code_samples.values())
embedded_99 = next((leg[99] for leg in embedded_legends.values() if 99 in leg), None)
if any_99_seen and embedded_99 and 99 in OPEN_WATER_CODES:
    findings.append(
        f'F3: Code 99 ({embedded_99!r} per embedded metadata) is observed in sampled pixels and '
        f'declared in OPEN_WATER_CODES, but no Atlantis output layer encodes it. The actual '
        f'permanent / open-water signal is therefore lost.'
    )

# F4 — Codes the embedded TIFF documents but ATBD + our pre-notebook notes ignore.
for bid, leg in embedded_legends.items():
    unknown_codes: list[tuple[int, str]] = []
    for code, label in leg.items():
        if code not in ATBD_LEGEND and code not in PUBLIC_TIFF_LEGEND:
            unknown_codes.append((code, label))
    if unknown_codes:
        pretty = ', '.join(f'{c}={lbl!r}' for c, lbl in unknown_codes)
        findings.append(
            f'F4: Embedded [{bid}] metadata declares codes that are missing from our '
            f'bundled ATBD and pre-notebook notes: {pretty}. These pass through Atlantis as '
            f'0.0 flood_fraction with no quality/water flag.'
        )

# F5 — Codes that disagree between ATBD and embedded TIFF (e.g. 27).
for bid, leg in embedded_legends.items():
    for code, embedded_label in leg.items():
        atbd_label = ATBD_LEGEND.get(code, '').lower()
        emb_lower = embedded_label.lower()
        if (
            atbd_label
            and emb_lower
            and atbd_label.split()[0] not in emb_lower
            and emb_lower.split()[0] not in atbd_label
        ):
            findings.append(
                f'F5: Code {code} disagrees between sources — ATBD={ATBD_LEGEND[code]!r}, '
                f'embedded [{bid}]={embedded_label!r}. Trust the embedded label.'
            )

if not findings:
    print('No automatic findings flagged (Atlantis matches the embedded NOAA TIFF legend).')
else:
    for f in findings:
        print('•', f)
        print()

### Visual sanity check — class distribution inside the event bbox

A quick bar chart so we can eyeball whether the codes we just discussed are dominant or marginal.

In [ ]:
import matplotlib.pyplot as plt

def _label_for(code: int) -> str:
    if code in embedded_legends.get('noaa_s3', {}):
        return f'{code} {embedded_legends["noaa_s3"][code]}'
    if 100 <= code <= 200:
        return f'{code} water-frac {code - 100}%'
    return f'{code} ?'

bid = 'noaa_s3' if 'noaa_s3' in code_samples else next(iter(code_samples), None)
if bid is None:
    print('No backend samples available to plot.')
else:
    s = code_samples[bid].sort_values(ascending=False)
    top = s.head(10)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.barh([_label_for(int(c)) for c in top.index][::-1], top.values[::-1])
    ax.set_xlabel(f'pixel count (Vamco bbox window, [{bid}])')
    ax.set_title(f'Top {len(top)} pixel codes — {bid} on {backend_dates[bid].date()}')
    for i, v in enumerate(top.values[::-1]):
        ax.text(v, i, f' {v:,}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## 9. Summary

The decision this notebook locks in: **the authoritative VIIRS legend is the embedded NOAA TIFF
metadata** — specifically the band tag `WaterDetection#TypeDescription` (in `rasterio`:
`ds.tags(1)['TypeDescription']`). It travels with the data, is maintained by the producer, and
in our test granule it reads:

```
1: no_valid_data; 15: Floodwater without fraction retrieval; 16: Bareland; 17: Vegetation;
20: Snow_ice; 27: River/lake ice; 30: Cloud; 38: Supar-snow/ice water or mixed ice&water or
melting ice; 50: Shadow; 99: NormalWater; 100-200: Water Fractions
```

Verified takeaways:

1. **Trust the NOAA embedded tag.** Both the bundled ATBD and our pre-notebook scraped notes
   contradict it on several codes. The notebook's findings cell flags every disagreement
   automatically.
2. **Code 17 is `Vegetation`** in the embedded tag. Atlantis maps `17 → permanent_water`. **Wrong.**
3. The flood-fraction range is **`100-200`** per the embedded tag, not `[101, 200]` as ATBD and
   `processor.py` use. Code 100 is silently dropped.
4. Atlantis declares `OPEN_WATER_CODES = {99}` but never writes it to any output layer — the
   `NormalWater` signal (6.6% of pixels inside the Vamco bbox here) is lost.
5. The embedded legend includes codes **16 (`Bareland`)** and **38 (mixed snow/ice/water)** that
   are absent from both ATBD and our notes; they pass through as `0.0` with no quality flag.
6. **Code 27** is contradictory: ATBD says `Bad input`, embedded NOAA says `River/lake ice`.
   Follow the embedded label.

**Out of scope here.** The `gmu_legacy` backend's `/downloads/pub` subtree is unreachable
(server-side 503; tracked on GitHub) so this notebook only validates `noaa_s3`. When the GMU
server returns we can simply append `'gmu_legacy'` to `ACTIVE_BACKENDS` and re-run.

The cross-check table in cell 7 and the findings in cell 8 are the evidence to reference when
fixing `src/atlantis/fetchers/viirs/processor.py`, `docs/viirs/overview.md`, `docs/viirs/api.md`,
and the notes in `tmp/source_layers_comparison.md` and `/memories/repo/viirs-public-codes.md`.